Importations et Configuration de l'environnement

In [1]:
import os
import time
import json
import logging
import requests
import pandas as pd
import numpy as np
from typing import List, Dict, Any

# Configuration du logging professionnel pour suivre l'exécution de l'ETL
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)

# Constantes de l'architecture des données
CISA_KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
NVD_API_BASE_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
DATA_DIR = "data"
CHECKPOINT_FILE = os.path.join(DATA_DIR, "nvd_api_raw_vulnerabilities.json")

# Création du répertoire de stockage si inexistant
os.makedirs(DATA_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Paramètres ajustables — regroupés ici pour itérer sur les décisions
# de data engineering (collecte, feature engineering, seuils) sans
# avoir à les rechercher dans tout le notebook.
# ------------------------------------------------------------------
# Périmètre recentré sur l'authentification et l'autorisation des API (cf. mémoire R1/R2/R3) :
# CWE-287 Improper Authentication, CWE-862 Missing Authorization, CWE-863 Incorrect Authorization,
# CWE-306 Missing Authentication for Critical Function, CWE-307 Improper Restriction of Excessive
# Authentication Attempts, CWE-798 Use of Hard-coded Credentials, CWE-613 Insufficient Session Expiration
TARGET_API_CWES = ["CWE-287", "CWE-862", "CWE-863", "CWE-306", "CWE-307", "CWE-798", "CWE-613"]
MAX_RESULTS_PER_PAGE = 2000
CRITICAL_FEATURES = ['attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction']
FEATURES_TO_ENCODE = ['attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope']
TEST_SIZE = 0.25
RANDOM_STATE = 42
# Seuil de décision fixe : une première intuition (Y=1 ≈ 0.8 % du corpus) à mettre à l'épreuve
# en Phase 4. Ce choix conditionne uniquement la matrice de confusion (precision/recall/F1) —
# l'AUC, métrique de rang, en est totalement indépendante et reste invariante quel que soit le seuil.
CLASSIFICATION_THRESHOLD = 0.05
EXPERIMENTS_LOG_FILE = os.path.join(DATA_DIR, "experiments_log.csv")


def log_experiment(notebook: str, phase: str, model: str, auc: float, threshold: float, n_test: int, **params) -> None:
    """
    Ajoute une ligne au journal d'expériences (data/experiments_log.csv).
    Permet de comparer AUC, seuils et paramètres entre itérations sans recopie manuelle.
    """
    entry = {
        "timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
        "notebook": notebook,
        "phase": phase,
        "model": model,
        "auc": round(float(auc), 4),
        "threshold": round(float(threshold), 6),
        "n_test": int(n_test),
        "params": json.dumps(params, ensure_ascii=False),
    }
    pd.DataFrame([entry]).to_csv(
        EXPERIMENTS_LOG_FILE,
        mode="a",
        header=not os.path.exists(EXPERIMENTS_LOG_FILE),
        index=False,
    )
    logging.info(f"[{model}] AUC={auc:.4f} | seuil={threshold:.6f} -> journalisé dans {EXPERIMENTS_LOG_FILE}")

Collecte de la Vérité Terrain (CISA KEV)

In [ ]:
def fetch_cisa_kev_dataset(url: str) -> set:
    """
    Télécharge le catalogue officiel CISA KEV et extrait l'ensemble 
    des identifiants CVE uniques activement exploités dans la nature.
    """
    logging.info("Initialisation du téléchargement du catalogue CISA KEV...")
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
         
        # Extraction des identifiants uniques (CVE-ID)
        vulnerabilities = data.get("vulnerabilities", [])
        cve_set = {vuln["cveID"] for vuln in vulnerabilities if "cveID" in vuln}
        
        logging.info(f"Catalogue CISA KEV ingéré avec succès. {len(cve_set)} CVE critiques identifiées.")
        return cve_set
    except Exception as e:
        logging.critical(f"Erreur fatale lors de la récupération des données CISA KEV: {e}")
        raise
        
# Exécution de l'ingénierie CISA
cisa_exploited_cves = fetch_cisa_kev_dataset(CISA_KEV_URL)

2026-06-15 14:14:49,035 [INFO] Initialisation du téléchargement du catalogue CISA KEV...
2026-06-15 14:14:51,466 [INFO] Catalogue CISA KEV ingéré avec succès. 1619 CVE critiques identifiées.


Collecte Massives et Résiliente depuis l'API NVD
###### L'API du NVD limite strictement le nombre de requêtes sans clé API (généralement 5 requêtes par tranche de 30 secondes). Le code ci-dessous implémente une temporisation stricte pour garantir qu'aucune requête ne soit rejetée avec une erreur HTTP 403 ou 429. Nous ciblons les faiblesses critiques de l'OWASP API Top 10 via une chaîne de requête CWE.

In [3]:
def fetch_nvd_api_vulnerabilities(base_url: str, target_cwes: List[str], max_results_per_page: int = 2000) -> List[Dict[str, Any]]:
    """
    Interroge l'API v2.0 du NVD de manière paginée en filtrant sur les CWE critiques des API Web.
    Gère nativement le rate-limiting pour une exécution stable en environnement de recherche.
    """
    # Réutilisation du checkpoint disque : évite de relancer une collecte de plusieurs
    # minutes (pagination rate-limitée) si CHECKPOINT_FILE existe déjà sur disque.
    # Supprimez ce fichier pour forcer une collecte fraîche depuis l'API NVD.
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            cached_items = json.load(f)
        logging.info(f"Checkpoint NVD détecté ({CHECKPOINT_FILE}) : {len(cached_items)} CVE rechargées depuis le disque.")
        return cached_items

    all_cve_items = []
    
    for cwe in target_cwes:
        start_index = 0
        total_results = 1 # Initialisation dynamique lors du premier appel
        logging.info(f"Début de l'extraction de l'API NVD pour la faiblesse : {cwe}")
        
        while start_index < total_results:
            params = {
                "cweId": cwe,
                "resultsPerPage": max_results_per_page,
                "startIndex": start_index
            }
            
            try:
                # Appel HTTP synchrone contrôlé
                response = requests.get(base_url, params=params, timeout=30)
                
                if response.status_code == 403 or response.status_code == 429:
                    logging.warning("Rate limit détecté par l'API NVD. Activation de la temporisation de secours (30s)...")
                    time.sleep(30)
                    continue
                    
                response.raise_for_status()
                payload = response.json()
                
                # Mise à jour du nombre total d'éléments existant pour cette CWE
                total_results = payload.get("totalResults", 0)
                vulnerabilities = payload.get("vulnerabilities", [])
                
                for item in vulnerabilities:
                    cve_data = item.get("cve", {})
                    all_cve_items.append(cve_data)
                
                logging.info(f"Progression [{cwe}] : {len(all_cve_items)}/{total_results} CVE capturées.")
                start_index += max_results_per_page
                
                # Temporisation de courtoisie réglementaire exigée par le NIST (6 secondes entre les appels)
                time.sleep(6)
                
            except Exception as e:
                logging.error(f"Erreur lors de la pagination à l'index {start_index} pour la CWE {cwe}: {e}")
                time.sleep(10) # Pause de sécurité avant reconnexions
                
    # Sauvegarde de sauvegarde (Checkpointing)
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(all_cve_items, f, ensure_ascii=False, indent=4)
        
    logging.info(f"Phase d'extraction NVD terminée. Fichier de sauvegarde généré : {CHECKPOINT_FILE}")
    return all_cve_items

# Exécution de l'extraction : réutilise le checkpoint si présent (sinon peut prendre
# plusieurs minutes — pagination rate-limitée). CWE ciblées et pagination définies
# dans les paramètres ajustables de la cellule précédente.
raw_nvd_data = fetch_nvd_api_vulnerabilities(NVD_API_BASE_URL, TARGET_API_CWES, MAX_RESULTS_PER_PAGE)

2026-06-15 14:14:55,890 [INFO] Checkpoint NVD détecté (data\nvd_api_raw_vulnerabilities.json) : 11146 CVE rechargées depuis le disque.


Parsing, Fusion Relatinnelle et Création du Dataset

In [4]:
def transform_and_merge_datasets(nvd_data: List[Dict[str, Any]], cisa_keys: set) -> pd.DataFrame:
    """
    Parse les structures de dictionnaires JSON imbriqués du NVD, extrait les métriques
    CVSS v3.1 vectorielles, identifie la ou les CWE associées à chaque CVE, et réalise
    la labellisation finale (Y) via l'ensemble CISA.
    """
    processed_records = []
    
    for cve in nvd_data:
        cve_id = cve.get("id")
        metrics = cve.get("metrics", {})

        # Extraction des CWE associées (champ "weaknesses") : trace explicitement le lien
        # entre chaque ligne et le périmètre Auth/Authz défini par TARGET_API_CWES (cellule 1).
        # Une CVE peut être rattachée à plusieurs CWE (entrées "Primary"/"Secondary") :
        # on conserve l'ensemble trié et joint en chaîne pour rester compatible CSV.
        cwe_ids = sorted({
            description.get("value")
            for weakness in cve.get("weaknesses", [])
            for description in weakness.get("description", [])
            if description.get("lang") == "en" and str(description.get("value", "")).startswith("CWE-")
        })
        
        # Recherche prioritaire des données normalisées CVSS v3.1
        cvss_v31_configs = metrics.get("cvssMetricV31", [])
        if not cvss_v31_configs:
            # Fallback vers CVSS v3.0 si la v3.1 est manquante pour maximiser le volume
            cvss_v31_configs = metrics.get("cvssMetricV30", [])
            
        if cvss_v31_configs:
            cvss_data = cvss_v31_configs[0].get("cvssData", {})
            
            record = {
                "cve_id": cve_id,
                "cwe_id": ",".join(cwe_ids),
                "base_score": cvss_data.get("baseScore"),
                "attack_vector": cvss_data.get("attackVector"),
                "attack_complexity": cvss_data.get("attackComplexity"),
                "privileges_required": cvss_data.get("privilegesRequired"),
                "user_interaction": cvss_data.get("userInteraction"),
                "scope": cvss_data.get("scope"),
                "confidentiality_impact": cvss_data.get("confidentialityImpact"),
                "integrity_impact": cvss_data.get("integrityImpact"),
                "availability_impact": cvss_data.get("availabilityImpact"),
                # Variable de labellisation mathématique (Y) : Appartenance à l'ensemble CISA KEV
                "is_exploited": 1 if cve_id in cisa_keys else 0
            }
            processed_records.append(record)
            
    df = pd.DataFrame(processed_records)
    # Suppression des doublons potentiels causés par les CVE multi-classes
    df.drop_duplicates(subset=["cve_id"], inplace=True)
    
    # Exportation finale du travail d'ingénierie
    output_path = os.path.join(DATA_DIR, "api_vulnerabilities_processed.csv")
    df.to_csv(output_path, index=False)
    logging.info(f"Dataset structuré créé avec succès. Forme de la matrice : {df.shape}")
    logging.info(f"Répartition des CVE par CWE cible :\n{df['cwe_id'].value_counts()}")
    logging.info(f"Distribution de la variable cible Y (is_exploited) :\n{df['is_exploited'].value_counts()}")
    return df

# Génération du DataFrame final
dataset_api = transform_and_merge_datasets(raw_nvd_data, cisa_exploited_cves)
dataset_api.head()

2026-06-15 14:15:18,727 [INFO] Dataset structuré créé avec succès. Forme de la matrice : (9995, 12)
2026-06-15 14:15:19,279 [INFO] Répartition des CVE par CWE cible :
cwe_id
CWE-862                    2608
CWE-287                    1737
CWE-863                    1186
CWE-798                     967
CWE-306                     945
                           ... 
CWE-20,CWE-613                1
CWE-601,CWE-613               1
CWE-524,CWE-613               1
CWE-269,CWE-384,CWE-613       1
CWE-521,CWE-613               1
Name: count, Length: 488, dtype: int64
2026-06-15 14:15:19,430 [INFO] Distribution de la variable cible Y (is_exploited) :
is_exploited
0    9911
1      84
Name: count, dtype: int64


,cve_id,cwe_id,base_score,attack_vector,attack_complexity,privileges_required,user_interaction,scope,confidentiality_impact,integrity_impact,availability_impact,is_exploited
0,CVE-2007-1966,CWE-287,9.1,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,NONE,0
1,CVE-2007-4043,CWE-287,9.8,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,0
2,CVE-2008-3738,CWE-287,9.1,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,NONE,0
3,CVE-2009-0130,CWE-287,7.5,NETWORK,LOW,NONE,NONE,UNCHANGED,NONE,HIGH,NONE,0
4,CVE-2009-1596,CWE-287,6.5,NETWORK,LOW,LOW,NONE,UNCHANGED,NONE,HIGH,NONE,0


In [5]:
import datetime
print("[+] Calcul de la feature temporelle : Âge de la CVE...")
# Si le DataFrame n'existe pas en mémoire (exécution partielle), on recharge le CSV traité
if 'df_vulnerabilities' not in globals():
    import os
    INPUT_PATH = os.path.join(DATA_DIR, "api_vulnerabilities_processed.csv")
    df_vulnerabilities = pd.read_csv(INPUT_PATH)

# Conversion des colonnes de dates au format datetime standard
df_vulnerabilities['published_date'] = pd.to_datetime(df_vulnerabilities['published_date'], errors='coerce')

# Référence temporelle fixe (date d'analyse) - ajuster si nécessaire
reference_date = pd.to_datetime('2026-06-15')

# Calcul de l'âge en jours
df_vulnerabilities['cve_age_days'] = (reference_date - df_vulnerabilities['published_date']).dt.days

# Traitement des valeurs manquantes par la médiane (imputation robuste)
median_age = df_vulnerabilities['cve_age_days'].median()
df_vulnerabilities['cve_age_days'] = df_vulnerabilities['cve_age_days'].fillna(median_age)

print(f"[+] Feature 'cve_age_days' créée avec succès. Médiane : {median_age} jours.")

# Écrasement du fichier traité sur disque
output_path = os.path.join(DATA_DIR, 'api_vulnerabilities_processed.csv')
df_vulnerabilities.to_csv(output_path, index=False)
print(f"[+] Fichier mis à jour : {output_path}")

[+] Calcul de la feature temporelle : Âge de la CVE...


KeyError: 'published_date'